In [1]:
import sys
import asyncio
import io

# Force Windows to use the Proactor Event Loop for async subprocesses
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Patch Jupyter's OutStream to prevent the fileno crash during process fallback
if hasattr(sys.stderr, "fileno"):
    try:
        sys.stderr.fileno()
    except io.UnsupportedOperation:
        sys.stderr.fileno = lambda: 2

In [ ]:
#create mcp ai agent to manipulate mongodb
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

True

In [6]:
import warnings 
warnings.filterwarnings(action="ignore", category=DeprecationWarning)


In [ ]:
from langchain.chat_models import init_chat_model

model_llama = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
model_qwen = init_chat_model("qwen-2.5-32b", model_provider="groq")
model_gem = init_chat_model("gemini-3.6-flash", model_provider="google_genai")


gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")
llm_primary = init_chat_model(model="openai/gpt-oss-120b", model_provider="Groq")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


In [8]:
#create mcp client & connect with mcp-mongodb server
from langchain_mcp_adapters.client import MultiServerMCPClient

In [9]:
import os

mongodb_uri = os.getenv("MONGODB_URI", "mongodb://localhost:27017")

client = MultiServerMCPClient(
    {
        "mongodb": {
            "transport": "stdio",
            "command": "node",
            "args": [
                r"C:\Users\ramma\AppData\Roaming\npm\node_modules\mongodb-mcp-server\dist\index.js"
            ],
            "env": {
                "PATH": os.environ.get("PATH", ""),
                "MDB_MCP_CONNECTION_STRING": mongodb_uri,
            },
        }
    }
)

In [10]:
import os
import subprocess

# Set your connection string or fallback
os.environ["MDB_MCP_CONNECTION_STRING"] = os.getenv(
    "MONGODB_URI", "mongodb://localhost:27017"
)

# Run via npx directly and capture stderr
cmd = ["npx.cmd", "-y", "mongodb-mcp-server@latest"]
res = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=os.environ,
    shell=True,
)

print("--- STDOUT ---")
print(res.stdout)
print("--- STDERR ---")
print(res.stderr)

--- STDOUT ---

--- STDERR ---
node:internal/modules/esm/resolve:229
    throw new ERR_UNSUPPORTED_DIR_IMPORT(path, basePath, String(resolved));
          ^

Error [ERR_UNSUPPORTED_DIR_IMPORT]: Directory import 'C:\Users\ramma\AppData\Local\npm-cache\_npx\57c35de36e7d43c4\node_modules\zod\v4' is not supported resolving ES modules imported from C:\Users\ramma\AppData\Local\npm-cache\_npx\57c35de36e7d43c4\node_modules\@modelcontextprotocol\core\dist\schemas-DscsoNZN.mjs
    at finalizeResolution (node:internal/modules/esm/resolve:229:11)
    at moduleResolve (node:internal/modules/esm/resolve:846:10)
    at defaultResolve (node:internal/modules/esm/resolve:973:11)
    at #cachedDefaultResolve (node:internal/modules/esm/loader:708:20)
    at #resolveAndMaybeBlockOnLoaderThread (node:internal/modules/esm/loader:728:38)
    at ModuleLoader.resolveSync (node:internal/modules/esm/loader:766:56)
    at #resolve (node:internal/modules/esm/loader:690:17)
    at ModuleLoader.getOrCreateModuleJob 

In [11]:
mongo_tools = await client.get_tools()

In [12]:
print("Discovered tools:", sorted(tool.name for tool in mongo_tools))

Discovered tools: ['aggregate', 'aggregate-db', 'atlas-local-connect-deployment', 'atlas-local-create-deployment', 'atlas-local-delete-deployment', 'atlas-local-list-deployments', 'collection-indexes', 'collection-schema', 'collection-storage-size', 'connect', 'count', 'create-collection', 'create-index', 'db-stats', 'delete-many', 'disconnect', 'drop-collection', 'drop-database', 'drop-index', 'explain', 'export', 'find', 'insert-many', 'list-collections', 'list-connections', 'list-databases', 'list-knowledge-sources', 'mongodb-logs', 'rename-collection', 'search-knowledge', 'update-many']


In [13]:
tool_names = {"count", "find", "insert-many", "update-many", "delete-many"}
lab_tools = [tool for tool in mongo_tools if tool.name in tool_names]


missing = tool_names - {tool.name for tool in lab_tools}
if missing:
    raise RuntimeError(f"Missing MongoDB tools: {sorted(missing)}")

from langchain.agents import create_agent
agent = create_agent(
    model=model_gem,
    tools=lab_tools,
    system_prompt=(
        "Use MongoDB tools only on the university database and students collection. "
        "Perform only the requested operation. For updates and deletes, require "
        "an exact _id filter targeting one document. Report the actual tool result."
    ),
)


async def run_query(query: str) -> None:
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": query}]}
    )
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            print("Tool used:", call["name"])
    print(result["messages"][-1].content)

In [15]:
def clean_schema(schema: dict) -> dict:
    """Recursively strip unsupported JSON Schema keys for Gemini compatibility."""
    if not isinstance(schema, dict):
        return schema
    
    # Keys that trigger Gemini / OpenAPI schema warnings
    unsupported_keys = {"$schema", "additionalProperties"}
    cleaned = {k: clean_schema(v) for k, v in schema.items() if k not in unsupported_keys}
    return cleaned

# Clean args_schema for every MCP tool
for t in lab_tools:
    if hasattr(t, "args_schema") and isinstance(t.args_schema, dict):
        t.args_schema = clean_schema(t.args_schema)

In [16]:
user_query = (
    "How many documents are in the students collection "
    "in the university database? Use the count tool."
)


await run_query(user_query)

Tool used: count
[{'type': 'text', 'text': 'There are 3 documents in the `students` collection in the `university` database.', 'extras': {'signature': 'EqUBCqIBAWkUfRPfMs52s8BUesQY7eW9DkHWYdY0lzHfnsEFBOGaBTUHkb+Hm7TenW+PvMmFh1LFTSw9SL6PufnZ3etNvYY9hs8478IukYGYwdBdLj/S5WcsPlXnl92unFn8LTfK00xN1IWKMXyjDHUFTllkKVSvFDhhuZhsMa+GsPrjTwJk2o7WMxe0denc3caaCZTxV9N/F0wsLV2gWjGx+n7N8CXt'}}]


In [ ]:
# create 
user_query = ("""
Insert one document into the students collection in the university database with _id S104, name Ethan Nawaz, dept_name Computer Science, 
gpa 3.99, and credit_hours 112. Store GPA and credit hours as numbers.
""")

await run_query(user_query)


Tool used: insert-many
[{'type': 'text', 'text': 'Documents were inserted successfully.\n\n**Tool Output:**\nInserted `1` document(s) into `university.students`.\nInserted IDs: `S104`', 'extras': {'signature': 'EoMCCoACAWkUfRP3dIROyeJ3PuUQr59JDDqT0OZlIpGen95Umbt3o4POWQChgtbcJ5AnsQCO4il+id3L2tvyhIS6Bsq0G6xU6GFvxGvV/+WQ8eezZY19qWtBHOudnHdvOEIwAdxDrH/qyKqzon5V2zEjjg6WPfAcNute4g9C1cfpk0APOCcQ1ATIS1Jit2MJgKd2OytiOcrC/HAD4rPrQfQzvdBOsB45jxXXlgXqSJiXS8KN/POuGWfPdFptBM1xXl8tYpmR0HPXuNI8swRpbEr2+AEsH63Zw1oU4zX26WL+MrtQ/CDyKsRESdY4SJQXxyPtXVMeaghwJKXH0LwR1Oo9djDWug=='}}]


In [18]:
#read
user_query=("""
Find S104 and return the student’s name, department, GPA, and credit hours.
Database:university, collection: students
""")
await run_query(user_query)

Tool used: find
Tool used: find
[{'type': 'text', 'text': 'Here are the details for student **S104**:\n\n- **Name:** Ethan Nawaz\n- **Department:** Computer Science (`dept_name`: Computer Science)\n- **GPA:** 3.99\n- **Credit Hours:** 112', 'extras': {'signature': 'EoADCv0CAWkUfROnTUfj5ayzrAyDR+jvBs95EgaBjGEGefHojOpJ3P4Zqf2IDZOnVl7GhXZAkuNsU/zjioib3u1jVfmaBQ+KLy92bPw6SDkmvJ2s83jWeMBvuhuNoq/CWLMC1XPUaFm680xJ+nC8p+uWwH9D5ILlJGJVh6MurzcC/a0VhE50FaDMlYF0NAscr41xHffVc7xxVj16h8HcavCOb7JkNHQ2U0VERbw6xazva64La4+W+9mpeRj6KuC8EzPxcBI9kMBC5vxG1RUchmbALk6UxehXqoXZIBbAvz8aDhd5yQv44CDErO8A+11lF7Ua35Hf0CeoYbCHPuVT0XGSJ+xyGBnatLwqSqnkCwCVJNMWPD3TmFTXlCkOQULvg9iRE9xUF3cHjmfQRMs7LMjDzT4tfmqfaU/r2MJJvXDsOhWsilJKQVs9i6wkqiOg1rk98e/e7Fty91G3/SJHHExrKsvID0V7o2iMZF9z2v1So/pVuhuZLtkL+lquhhnFyoTD'}}]


In [19]:
#match and update 
user_query=("""
Match S104 by its exact _id and change only credit_hours from 112 to 115.
Database:university, collection: students
""")

await run_query(user_query)

Tool used: find
Tool used: find
Tool used: find
Tool used: update-many
[{'type': 'text', 'text': 'The requested update operation has been performed.\n\n**Tool Execution Result:**\n`Matched 1 document(s). Modified 1 document(s).`\n\nDocument `S104` in `university.students` now has `credit_hours` updated to `115`.', 'extras': {'signature': 'EpUFCpIFAWkUfRNYx7CPiYJLkq2Aohmdv98jLnSeWXv7XWCfoVZFbQ9xY9x2MPrMc+s8a57CuELJl1I1Rphu/y5hcBaCTm/2waL0cNM3y3aG1Io5FZ6VRjqbonvdQGgKDKsRGd9PmoAQ0CKJHXZXIy0wOgInk3RYK9nMmOarXKHa57LMz/6piQdDYTCQkffqlHaOyhtZ/JpcGymQ9xORMeC6WR3H7L/pcIwDXhJy7Y8qY8ye8x9SXjr4R6jtW3A6W9X+jNMAkgTmPU4tZ7EUesgBMP1BQCs7QC3y5gvfv5YLFXkBgWHImNVzxWjYdrD3eyjXt3xmjs0Zc1Kop/If/neZcaKaVbI4B7Zyop6D00j6K8TfZL/0FsMB0EhcRx3VASK4tz3FkHPyHRdXDEaL3TwmfIzX0jNDNpCAc/MVSvac+gwM881w+8tnM4KM8X7T4lkQp6ytqQjSrt/in4eV4nyq81Idv/XqpCnOctWGKOKHlIujaLkyWdUHTr0wpKwd4ZGRKrbUwD8tdi8wGrzp+lS12ZYUwfB7Q9wo1TR53qGcZY2Z9jgvqGMB2dSzo5j9ZlJc6scmsKHymZ1pGOPgmZWmgy5dKZDLWOhKJ8hdfKZ3hNurVTEnXEy2u7VDecNk/MGdEvGc9YV+UKz1Ptj

In [20]:
user_query=("""
Delete only the document whose _id is S104.
Database: university, collection: students
""")
await run_query(user_query)

Tool used: delete-many
Tool used: delete-many
[{'type': 'text', 'text': 'Deleted 1 document(s) from the requested collection.', 'extras': {'signature': 'Eo8CCowCAWkUfRNj7qL4E/P1oVf05dUlN/dFI196JryezWWgZFVm0clJdzGGvLzL+smGL9eThQrskQ1CJS99Pdop0IwQIxWqwX5co0/Qb6Jw/sDiXFgau9nNkbpL22VGJsA9ceIjPaQuZ0ofeDsmR2gqVw4XfkPuMTPCAEeec09XrmQnplQP0v1GVQj/8OeOnvSvMVOjToKT1ClhlrbZE1U+GlnVhbjFmOHd48ciHxcJKdf+Dh6vc+VXqikWYr1D2q332InoPSU3eNVRr2tOjtRviJ4ipltjn0EcW27CfPN7Cfsj+MzraTkzNCNwTivGpRRYiU2l66j8Fy5uLjtLKNjWy2QKR4QsE/aeh1MAV1xhsg=='}}]


In [21]:
user_query = (
    "How many documents are in the students collection "
    "in the university database? Use the count tool."
)
await run_query(user_query)

Tool used: count
[{'type': 'text', 'text': 'There are 3 documents in the `students` collection in the `university` database.', 'extras': {'signature': 'EtkBCtYBAWkUfRNhlWEWRaRMsF36csAc2wMe/La7j5GqNgqCjdc5m9PvfcADeCLdqilAoHLKYUPldEo3/0pYw+ABsJmhr8SDlP0qVJADtsD1XIc46IrxC3FwW2v7Z4nxFbiv5MEwpIx6pMA9a2sVgCrm4qQJ0Fa+6DAq3EWeHcycAFAAAsxdKwtiW/ogE0CJ/DmUo0Wb+yKlmS0m9wCSk9UttpV6KKrzi70BKwvnvshydOFH3cBvYrkV1u2JC0IRyyQ7o7XQ6Zp9LcVqkBjwWT5CR48GpAasYX/jAw=='}}]
